# Stillwater SewerTris Example

This notebook implements the Stillwater SewerTris workflow using the refactored `sewertris` package in `src/sewertris`. It starts from the Stillwater domain shapefile, fills the domain with tetromino-style urban blocks, generates roads and topography, builds a sanitary sewer network, assigns land use and design flows, exports an EPA-SWMM model, defines BWF, GWI, and RDII inputs, and saves decomposed flow outputs for analysis.

The notebook is organized around the twelve key SewerTris model implementation steps:

1. Urban Domain Definition: Defines the spatial modeling boundary using a vector polygon or raster mask and establishes the sewer outlet location.

2. Tetris Block Definition: Specifies modular tetromino building blocks (I, L, T, S, Z shapes) that form the geometric basis of the synthetic urban layout.

3. Stochastic Tetris Completion: Populates the domain using randomized block placement to generate heterogeneous but coherent urban configurations.

4. Road Network Extraction: Derives a synthetic road network from block boundaries, ensuring topological consistency with urban structure.

5. Land-Use Assignment: Assigns residential, commercial, industrial, public, and recreational land uses using rule-based or user-defined allocation strategies.

6. Synthetic DEM Generation: Creates a hydraulically consistent Digital Elevation Model (DEM) enforcing global drainage toward the outlet.

7. Sewer Network Generation: Constructs a gravity-driven, tree-structured sewer network aligned with roads and embedded within the DEM.

8. Sewer Flow Predesign: Computes baseline peak discharges combining Dry-Weather Flow (DWF), Groundwater Infiltration (GWI), and Rainfall-Derived Inflow and Infiltration (RDII).

9. Pipe Sizing and Hydraulic Properties: Assigns pipe diameters, roughness, and invert elevations using Manning-based design principles.

10. Dynamic Flow Input Definition: Specifies temporally resolved DWF, GWI, and RDII inputs, including rainfall forcing and spatial heterogeneity options.

11. EPA-SWMM Simulation: Performs unsteady hydraulic routing and enables component tagging (RAIN and DRY) for flow separation analysis.

12. Flow Output Decomposition: Extracts and decomposes total flows into DWF, RDII, and residual GWI components for benchmarking and diagnostics.


## Setup: Import Libraries

This cell imports the scientific Python stack and adds the repository `src` directory to `sys.path`, so the notebook uses the refactored `sewertris` package directly.


In [ ]:
import os
import random
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris as sp


## 1. Urban Domain Definition

Defines the spatial modeling boundary using the Stillwater domain shapefile in `input/domain_mask.shp`. The shapefile is rasterized into a model domain mask while preserving geospatial metadata for later export.


In [ ]:
# --- Parameters ---
min_width = 100  # meters
use_shapefile = True
shapefile_path = Path("input/domain_mask.shp")
if not shapefile_path.exists():
    shapefile_path = PROJECT_ROOT / "examples" / "input" / "domain_mask.shp"

if use_shapefile:
    gdf_in = gpd.read_file(shapefile_path)
    cell_size = sp.meters_to_crs_units(min_width, gdf_in.crs)
    domain_mask, grid_meta = sp.build_domain_mask_from_shapefile(
        shapefile_path,
        cell_size_m=cell_size,
    )
else:
    domain_mask = np.array([
        [0,0,0,0,0,1,1,1,0,0,1,1,1,1,1,0,0],
        [0,0,0,1,1,1,1,1,0,0,1,1,1,1,1,0,0],
        [0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,0],
        [0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0],
        [0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0],
        [0,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0],
        [0,0,1,1,1,1,1,1,0,1,1,1,1,1,0,0,0],
        [0,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,0],
        [0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1],
        [1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1],
    ])
    grid_meta = dict(
        crs_out="EPSG:3857",
        origin_x=0,
        origin_y=0,
        cell_w=min_width,
        cell_h=min_width,
        rows=domain_mask.shape[0],
        cols=domain_mask.shape[1],
        flip_y=False,
    )

sp.plot_domain_mask(domain_mask, title="Stillwater SewerTris Domain Mask")


## 2. Tetris Block Definition

Specifies modular tetromino building blocks (I, L, T, S, Z shapes) that form the geometric basis of the synthetic urban layout.


In [ ]:
# Tetromino definitions (all rotations)
tetrominoes = {
    'I': [np.array([[1,1,1,1]]), np.array([[1],[1],[1],[1]])],
    'O': [np.array([[1,1],[1,1]])],
    'T': [np.array([[1,1,1],[0,1,0]]), np.array([[0,1],[1,1],[0,1]]), np.array([[0,1,0],[1,1,1]]), np.array([[1,0],[1,1],[1,0]])],
    'S': [np.array([[0,1,1],[1,1,0]]), np.array([[1,0],[1,1],[0,1]])],
    'Z': [np.array([[1,1,0],[0,1,1]]), np.array([[0,1],[1,1],[1,0]])],
    'J': [np.array([[1,0,0],[1,1,1]]), np.array([[1,1],[1,0],[1,0]]), np.array([[1,1,1],[0,0,1]]), np.array([[0,1],[0,1],[1,1]])],
    'L': [np.array([[0,0,1],[1,1,1]]), np.array([[1,0],[1,0],[1,1]]), np.array([[1,1,1],[1,0,0]]), np.array([[1,1],[0,1],[0,1]])],
    'BO': [np.array([[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]])],
}

# Assign a color to each tetromino
tetromino_colors = {
    'I': 'cyan',
    'O': 'yellow',
    'T': 'purple',
    'S': 'green',
    'Z': 'red',
    'J': 'blue',
    'L': 'orange',
    'BO': 'grey',
}

sp.plot_tetromino_set(tetrominoes, tetromino_colors, ncols=4)


## 3. Stochastic Tetris Completion

Populates the Stillwater domain using randomized block placement to generate a coherent synthetic urban configuration, then exports the resulting blocks as geospatial polygons.


In [ ]:
layout_seed = 1000 # Seed 1002 is good for testing bad convergance in previous method
random.seed(layout_seed)
np.random.seed(layout_seed)

# Fill the domain with tetrominoes and blocks
filled_board, id_type_map, block_id = sp.fill_domain_with_tetrominoes_and_blocks(
    domain_mask,
    tetrominoes,
)
print("Total blocks:", len(np.unique(filled_board)))

# Export the filled board to a shapefile in the same coordinate space as the domain shapefile
output_folder = EXAMPLES_DIR / "output_example_2"
os.makedirs(output_folder, exist_ok=True)
output_shapefile = os.path.join(output_folder, "City_2.shp")

sp.export_individual_figures_to_shapefile_georeferenced(
    filled_board,
    output_shapefile,
    grid_meta,
    id_to_type_map=id_type_map,
)

sp.plot_filled_board_shapefile(output_shapefile)


## 4. Road Network Extraction

Derives a synthetic road network from block boundaries, ensuring topological consistency with the generated urban structure.


In [ ]:
# Generate road network from blocks
road_width = 10  # meters
road_lines, road_buffer, crs = sp.generate_road_network_from_blocks(
    blocks_path=output_shapefile,
    road_width=road_width,
    simplify_tol=0.5,
)

gpd.GeoDataFrame(geometry=[road_lines], crs=crs).to_file(
    os.path.join(output_folder, "road_centerlines.shp")
)
gpd.GeoDataFrame(geometry=[road_buffer], crs=crs).to_file(
    os.path.join(output_folder, "road_polygons.shp")
)

sp.plot_roads(
    road_lines=road_lines,
    road_buffer=road_buffer,
    crs=crs,
    title=f"Road polygons (width={road_width} m)",
)


## 5. Land-Use Assignment

Assigns residential, commercial, industrial, public, and recreational land uses using compact rule-based allocation.


In [ ]:
# Cut, label blocks and assign land use
blocks_path = output_shapefile
roads_path = os.path.join(output_folder, "road_polygons.shp")
output_path = os.path.join(output_folder, "City_2.shp")

blocks, road_network, crs = sp.load_blocks_and_roads(blocks_path, roads_path)
blocks = sp.cut_blocks(blocks, road_network)
blocks = sp.assign_land_use_compact(blocks)
gdf = sp.export_to_shapefile(blocks, crs, output_path)

roads_gdf = gpd.read_file(roads_path)
if roads_gdf.crs != gdf.crs:
    roads_gdf = roads_gdf.to_crs(gdf.crs)

sp.plot_blocks_landuse(
    blocks_gdf=gdf,
    roads_gdf=roads_gdf,
    landuse_col="land_use",
    title="Stillwater blocks with assigned land use",
    savepath=os.path.join(output_folder, "blocks_landuse.png"),
)


## 6. Synthetic DEM Generation

Creates a hydraulically consistent Digital Elevation Model (DEM) enforcing global drainage toward the configured outlet direction.


In [ ]:
# Extract road boundaries
roads_path = os.path.join(output_folder, "road_polygons.shp")
sp.extract_boundary(
    roads_path,
    out_boundary_lines=os.path.join(output_folder, "road_boundary_lines.shp"),
    out_outer_shell_polygon=os.path.join(output_folder, "road_outer_shell.shp"),
    keep_holes=False,
)

config = sp.TopographyConfig(
    min_elevation=270,
    max_elevation=290,
    cell_size=10,
    outlet_direction='S',
    smoothing_factor=1,
)

boundary_path = os.path.join(output_folder, "road_outer_shell.shp")
roads_path = os.path.join(output_folder, "road_polygons.shp")

if not os.path.exists(boundary_path) or not os.path.exists(roads_path):
    raise FileNotFoundError("Input shapefiles not found. Please check the file paths.")

try:
    elevation, xx, yy, mask = sp.generate_topography(boundary_path, roads_path, config)

    boundary_gdf = gpd.read_file(boundary_path)
    input_crs = boundary_gdf.crs

    if input_crs is None:
        print("Warning: Input CRS is None, defaulting to UTM Zone 14N (EPSG:32614)")
        input_crs = "EPSG:32614"

    print(f"Using CRS: {input_crs}")

    output_path = os.path.join(output_folder, "generated_topography.tif")
    transform = rasterio.transform.from_bounds(
        west=xx[0,0],
        south=yy[-1,0],
        east=xx[0,-1],
        north=yy[0,0],
        width=elevation.shape[1],
        height=elevation.shape[0],
    )

    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=elevation.shape[0],
        width=elevation.shape[1],
        count=1,
        dtype=elevation.dtype,
        crs=input_crs,
        transform=transform,
        nodata=np.nan,
    ) as dst:
        dst.write(elevation, 1)

    print(f"Topography successfully generated and saved to {output_path}")
    print(f"DEM resolution: {config.cell_size} m")
    print(f"Elevation range: {config.min_elevation} m - {config.max_elevation} m")
    print(f"Output CRS: {input_crs}")
    print("Grid extent:")
    print(f"  X: {xx[0,0]:.2f} to {xx[0,-1]:.2f}")
    print(f"  Y: {yy[0,0]:.2f} to {yy[-1,0]:.2f}")

except Exception as e:
    print(f"Error generating topography: {str(e)}")
    raise


## 7. Sewer Network Generation

Constructs a gravity-driven, tree-structured sewer network aligned with roads and embedded within the DEM.


In [ ]:
road_axes_path = os.path.join(output_folder, "road_centerlines.shp")
dem_path = os.path.join(output_folder, "generated_topography.tif")
manholes = sp.extract_manholes_from_lines(road_axes_path, dem_path)
sp.export_manholes_to_shapefile(
    manholes,
    os.path.join(output_folder, "manholes.shp"),
    crs=gpd.read_file(road_axes_path).crs,
)

road_buffer = road_lines.buffer(road_width * 0.6)

segments, path_info, graph_data = sp.generate_main_sewer_path_optimized(
    manholes=manholes,
    road_buffer=road_buffer,
    block_size=min_width * 2,
    slope_tolerance=-0.01,
    min_pipe_length=5.0,
    prefer_slope=0.5,
    return_graph_data=True,
)

main_path = path_info["segments"]

secondary_pipes, secondary_attrs = sp.generate_secondary_pipes_optimized(
    manholes=manholes,
    main_path=main_path,
    road_buffer=road_lines.buffer(road_width * 0.6),
    block_size=min_width * 2,
    slope_tolerance=0.00,
    prefer_slope=0.5,
    return_attrs=True,
)

secondary_pipes_clean = sp.remove_secondary_pipes_overlapping_main_optimized(
    manholes=manholes,
    secondary_pipes=secondary_pipes,
    main_pipes=main_path,
)

network_status = sp.build_current_network_status(
    manholes=manholes,
    main_path=main_path,
    secondary_pipes=secondary_pipes_clean,
)

print("Main outlet ID:", network_status["main_outlet_id"])
print("Total manholes:", len(network_status["all_ids"]))
print("Nodes already touched by main+secondary:", len(network_status["nodes_in_network"]))
print("Manholes missing outlet pipe:", len(network_status["missing_outlet_ids"]))
print("First 20 missing:", network_status["missing_outlet_ids"][:20])
print("Duplicate sources:", network_status["duplicate_sources"][:20])

tertiary_pipes, tertiary_unconnected, tertiary_attrs = sp.generate_tertiary_pipes_backtracking_stop_at_each_manhole(
    manholes=manholes,
    main_path=main_path,
    secondary_pipes=secondary_pipes_clean,
    road_buffer=road_lines.buffer(road_width * 0.6),
    city_boundary=os.path.join(output_folder, "road_outer_shell.shp"),
    block_size=min_width * 10,
    neighbor_radius_factor=1.5,
    min_pipe_length=1e-3,
    point_on_line_tol=0.01,
    return_attrs=True,
    max_search_depth=300,
)

main_attrs = sp.build_main_attrs_from_path_info(path_info)

gdf_pipes = sp.export_pipes_to_shapefile_2(
    pipes_main=main_path,
    pipes_sec=secondary_pipes_clean,
    pipes_ter=tertiary_pipes,
    manholes=manholes,
    output_path=os.path.join(output_folder, "sewer_pipes.shp"),
    crs=crs,
    main_attrs=main_attrs,
    secondary_attrs=secondary_attrs,
    tertiary_attrs=tertiary_attrs,
)

print("Sewer network generation complete.")

sp.plot_sewer_network_all(
    manholes=manholes,
    main_pipes=main_path,
    secondary_pipes=secondary_pipes_clean,
    tertiary_pipes=tertiary_pipes,
    unresolved=tertiary_unconnected,
    road_buffer=road_buffer,
    title="Stillwater SewerTris sewer network",
)


### Step 7 Continued: Embed the Sewer Network in the DEM

The DEM is adjusted along pipe alignments so the generated sewer network maintains positive gravity slopes.


In [ ]:
# Modify topography to follow the sewer network
print("Modifying topography to follow sewer network...")

dem_path = os.path.join(output_folder, "generated_topography.tif")
sewer_path = os.path.join(output_folder, "sewer_pipes.shp")
manholes_path = os.path.join(output_folder, "manholes.shp")
output_path = os.path.join(output_folder, "generated_topography.tif")

out_path = sp.build_dem_with_guaranteed_positive_slopes_idw(
    dem_path=dem_path,
    pipes_path=sewer_path,
    manholes_path=manholes_path,
    output_path=output_path,
    upstream_field="upstream_m",
    downstream_field="downstream",
    manhole_id_field="id",
    manhole_elev_field="elevation",
    type_field="type",
    tier_order=("main", "secondary", "tertiary"),
    Smin=0.001,
    densify_step_m=None,
    along_pipe_weight=2,
    idw_power=2.0,
    idw_k=12,
    idw_tile=1024,
    centerline_writeback=True,
    verify_on_raster=True,
)
print("Wrote:", out_path)

sp.update_manhole_elevations_from_dem(
    dem_path=out_path,
    manholes_path=manholes_path,
    output_path=None,
    overwrite=True,
    sampling="nearest",
)

sp.plot_dem_tif(output_path, title="Modified Stillwater Topography (DEM)", hillshade=False)


## 8. Sewer Flow Predesign

Computes baseline peak discharges combining Dry-Weather Flow (DWF), Groundwater Infiltration (GWI), and Rainfall-Derived Inflow and Infiltration (RDII).


In [ ]:
blocks_path = os.path.join(output_folder, "City_2.shp")
manholes_path = os.path.join(output_folder, "manholes.shp")
pipes_path = os.path.join(output_folder, "sewer_pipes.shp")
topography_path = os.path.join(output_folder, "generated_topography.tif")

# For each land use category:
# - density is in people per hectare [people/ha]
# - demand is in liters per person per day [L/person/day]
LAND_USE_INFO = {
    'RESIDENTIAL': {
        'density': 60,
        'demand': 100,
    },
    'COMMERCIAL': {
        'density': 50,
        'demand': 60,
    },
    'INDUSTRIAL': {
        'density': 25,
        'demand': 150,
    },
    'PUBLIC': {
        'density': 20,
        'demand': 100,
    },
    'RECREATIONAL': {
        'density': 10,
        'demand': 40,
    },
}

sp.delineate_afferent_areas_and_baseflow(
    blocks_path=blocks_path,
    pipes_path=pipes_path,
    manholes_path=manholes_path,
    topo_path=topography_path,
    output_path=os.path.join(output_folder, "sewer_subcatchments.shp"),
    land_use_info=LAND_USE_INFO,
)

subcatchments_path = os.path.join(output_folder, "sewer_subcatchments.shp")

sp.assign_flow_to_pipes_fast(
    pipes_path=pipes_path,
    subcatchments_path=subcatchments_path,
    output_path=os.path.join(output_folder, "sewer_pipes.shp"),
)

pipes = gpd.read_file(os.path.join(output_folder, "sewer_pipes.shp"))
peak_flow, pf = sp.british_columbia_peaking_factor(pipes["cumulative"])
pipes["peaking_factor_bc"] = pf
pipes["peak_flow_lps_bc"] = peak_flow
pipes.to_file(os.path.join(output_folder, "sewer_pipes.shp"))
print("Peak flows added using BC method.")

pipes_path = os.path.join(output_folder, "sewer_pipes.shp")

sp.compute_gwi_cumulative(
    pipes_path=pipes_path,
    gwi_factor_ls_per_m=0.0002,
    out_path=os.path.join(output_folder, "sewer_pipes.shp"),
    id_field="pipe_id",
    up_field="upstream_m",
    down_field="downstream",
    length_field=None,
    target_crs_m="EPSG:3857",
)

sp.compute_rdii_and_accumulate(
    pipes_path=pipes_path,
    subcatch_path=subcatchments_path,
    rdii_factor_ls_per_m2=0.00002,
    pipe_id_field="pipe_id",
    up_field="upstream_m",
    down_field="downstream",
    sub_pipe_field="pipe_id",
    target_crs_m="EPSG:3857",
    out_pipes=os.path.join(output_folder, "sewer_pipes.shp"),
    out_subcatch=os.path.join(output_folder, "sewer_subcatchments.shp"),
)

sp.add_predesign_flow(
    pipes_path=os.path.join(output_folder, "sewer_pipes.shp"),
    out_path=os.path.join(output_folder, "sewer_pipes.shp"),
)


## 9. Pipe Sizing and Hydraulic Properties

Assigns pipe slopes, diameters, roughness values, materials, and invert elevations using design rules compatible with Manning-based hydraulic sizing.


In [ ]:
pipes_path = os.path.join(output_folder, "sewer_pipes.shp")

sp.assign_pipe_slopes(
    pipes_path=pipes_path,
    manholes_path=manholes_path,
    output_path=os.path.join(output_folder, "sewer_pipes.shp"),
    minimum_slope=0.005,
)

pipes_path = os.path.join(output_folder, "sewer_pipes.shp")

sp.assign_material_diameter_to_pipes(
    pipes_path=pipes_path,
    output_path=os.path.join(output_folder, "sewer_pipes.shp"),
    material_fractions={"PVC": 0.6, "CONCRETE": 0.3, "HDPE": 0.1},
    n_by_material={"PVC": 0.011, "CONCRETE": 0.013, "HDPE": 0.012},
    standard_diameters_mm=[200, 250, 300, 350, 400, 450, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000],
    minimum_diameter_mm=200,
)

pipes_path = os.path.join(output_folder, "sewer_pipes.shp")

sp.assign_invert_elevations(
    pipes_path=pipes_path,
    output_path=os.path.join(output_folder, "sewer_pipes.shp"),
    min_cover=1.4,
    min_slope=0.005,
    manhole_drop=0.05,
)

pipes_path = os.path.join(output_folder, "sewer_pipes.shp")

sp.preprocess_pipes_and_manholes(
    pipes_path=pipes_path,
    manholes_path=manholes_path,
    output_pipes_path=os.path.join(output_folder, "sewer_pipes.shp"),
    output_manholes_path=os.path.join(output_folder, "manholes.shp"),
)

final_pipes = os.path.join(output_folder, "sewer_pipes.shp")
final_mhs = os.path.join(output_folder, "manholes.shp")

sp.plot_final_design_color_by_diameter(
    pipes_path=final_pipes,
    manholes_path=final_mhs,
    blocks_path=blocks_path,
    diameter_field="diameter_m",
    manhole_color_field="elevation",
    linewidth=1.8,
)


## 10. Dynamic Flow Input Definition

Specifies temporally resolved BWF/DWF, GWI, and RDII inputs, including daily/hourly patterns, baseline infiltration, rainfall forcing, and subcatchment routing. The model is run after each major input layer so the response can be checked incrementally.


In [ ]:
# Export to SWMM INP format
# Ten days simulation
routing_step = "0:01:00"  # 1 minute
START_DATE = "01/01/1990"
END_DATE = "01/10/1990"

pipes_path = os.path.join(output_folder, "sewer_pipes.shp")
manholes_path = os.path.join(output_folder, "manholes.shp")
options = {
    "FLOW_UNITS": "LPS",
    "INFILTRATION": "CURVE_NUMBER",
    "FLOW_ROUTING": "KINWAVE",
    "LINK_OFFSETS": "DEPTH",
    "MIN_SLOPE": "0",
    "ALLOW_PONDING": "NO",
    "SKIP_STEADY_STATE": "NO",
    "START_DATE": START_DATE,
    "START_TIME": "00:00:00",
    "REPORT_START_DATE": START_DATE,
    "REPORT_START_TIME": "00:00:00",
    "END_DATE": END_DATE,
    "END_TIME": "00:00:00",
    "SWEEP_START": "01/01",
    "SWEEP_END": "12/31",
    "DRY_DAYS": "0",
    "REPORT_STEP": "00:15:00",
    "WET_STEP": "00:05:00",
    "DRY_STEP": "01:00:00",
    "ROUTING_STEP": routing_step,
    "RULE_STEP": "00:00:00",
    "INERTIAL_DAMPING": "PARTIAL",
    "NORMAL_FLOW_LIMITED": "BOTH",
    "FORCE_MAIN_EQUATION": "D-W",
    "VARIABLE_STEP": "0.75",
    "LENGTHENING_STEP": "0",
    "MIN_SURFAREA": "1.167",
    "MAX_TRIALS": "8",
    "HEAD_TOLERANCE": "0.0015",
    "SYS_FLOW_TOL": "5",
    "LAT_FLOW_TOL": "5",
    "MINIMUM_STEP": "0.5",
    "THREADS": "1",
}

sp.export_swmm_inp(
    pipes_path=pipes_path,
    manholes_path=manholes_path,
    output_path=os.path.join(output_folder, "sewer_model.inp"),
    options_dict=options,
)


In [ ]:
# BWF/DWF patterns
SWMM_INPUT_FILE = os.path.join(output_folder, "sewer_model.inp")

hourly = [
    0.5631, 0.4163, 0.3181, 0.3178, 0.4838, 0.8014,
    1.1337, 1.3239, 1.3321, 1.2512, 1.1900, 1.1703,
    1.1489, 1.1063, 1.0762, 1.0943, 1.1545, 1.2270,
    1.2931, 1.3362, 1.3175, 1.1978, 0.9890, 0.7576,
]

daily_vals = [1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2]

monthly_vals = [
    0.90, 0.95, 1.00, 1.05,
    1.10, 1.10, 1.05, 1.00,
    0.95, 0.90, 0.90, 0.90,
]

fairfax_weekend = [
    0.5631, 0.4163, 0.3181, 0.3178, 0.3568, 0.4068,
    0.6068, 0.7068, 0.8014, 1.1337, 1.3239, 1.3321,
    1.1489, 1.1063, 1.0762, 1.0943, 1.1545, 1.2270,
    1.2931, 1.1362, 1.0175, 0.8978, 0.8090, 0.7576,
]

sp.assign_all_dwf_patterns(
    inp_path=SWMM_INPUT_FILE,
    output_path=os.path.join(output_folder, "sewer_model.inp"),
    hourly_id="1",
    hourly_values=hourly,
    daily_id="2",
    daily_values=daily_vals,
    monthly_id="3",
    monthly_values=monthly_vals,
    weekend_id="4",
    weekend_values=fairfax_weekend,
)

print("SWMM INP file with BWF/DWF patterns created.")




In [ ]:
# GWI
inp_file = os.path.join(output_folder, "sewer_model.inp")

gwi_coefficient = 0.00001
sp.assign_inflow_from_pipe_length(
    inp_path=inp_file,
    output_path=os.path.join(output_folder, "sewer_model.inp"),
    coefficient=gwi_coefficient,
)

sp.plot_inflow_from_pipe_length(
    inp_path=os.path.join(output_folder, "sewer_model.inp"),
    coefficient=gwi_coefficient,
)




In [ ]:
# RDII
inp_file = os.path.join(output_folder, "sewer_model.inp")
rainfall_seed = 42 # Seed 44 is good for testing bad convergance in previous method


rainfall_data = sp.generate_clustered_rainfall_timeseries(
    start_date=f"{START_DATE} 00:00",
    end_date=f"{END_DATE} 00:00",
    timestep_minutes=15,
    avg_annual_precip_mm=1200,
    wet_season_months=[4, 5, 6, 9, 10, 11],
    dry_wet_ratio=0.2,
    storm_prob=0.1,
    storm_duration_range=(1, 6),
    random_seed=rainfall_seed,
    preview_date=f"{START_DATE}",
)

subcatchments_path = os.path.join(output_folder, "sewer_subcatchments.shp")

sp.add_subcatchment_data_to_inp(
    inp_path=inp_file,
    output_path=os.path.join(output_folder, "sewer_model.inp"),
    subcatchments_path=subcatchments_path,
    raingage_id="1",
    raingage_coords=(500, 500),
    timeseries=rainfall_data,
    n_imperv=0.011,
    n_perv=0.15,
    s_imperv=0.0,
    s_perv=0.0,
    pct_zero=0,
    route_to="OUTLET",
    pct_routed="",
    infiltration_params=(90, 0.5, 7, "", ""),
    imperv_pct=5,
    width=100,
    slope=0.005,
    curblen=0,
)


## 11. EPA-SWMM Simulation

Adds pollutant/component tags to the SWMM input and runs PySWMM-based flow separation on the outlet pipe.


In [ ]:
inp_file = os.path.join(output_folder, "sewer_model.inp")

sp.auto_add_pollutants_to_inp_fixed(
    inp_path=inp_file,
    output_path=os.path.join(output_folder, "sewer_model.inp"),
)

inp_path = os.path.join(output_folder, "sewer_model.inp")

df = sp.get_flow_components_from_node_pyswmm(
    inp_path=inp_path,
    link_id="P_OUTLET",
)


## 12. Flow Output Decomposition

Extracts and decomposes total flows into DWF, RDII, and residual GWI components for benchmarking and diagnostics, then saves the flow dataset for later analysis.


In [ ]:
sp.plot_flow_components_v2(
    df,
    start=f"{START_DATE} 00:00:00",
    end=f"{END_DATE} 00:00:00",
)

flows_for_export = df.set_index("Datetime") if "Datetime" in df.columns else df
flows_ds = flows_for_export.to_xarray()
flows_ds.to_netcdf(os.path.join(output_folder, "flows.nc"))
print("Saved as flows.nc")
